# Create components for methods figure

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import contextily as ctx
import matplotlib.pyplot as plt
import xyzservices as xyz
import cartopy.crs as ccrs
from global_snowmelt_runoff_onset.config import Config, Tile
import easysnowdata
import global_snowmelt_runoff_onset.processing as processing
import rioxarray as rxr

In [ ]:
config = Config('config/global_config_v9.txt')

In [ ]:
global_hillshade_robinson_da = rxr.open_rasterio('../data/global_hillshade_robinson.tif', masked=True, chunks='auto').squeeze().coarsen(x=30, y=30, boundary='trim').mean().compute()
global_hillshade_robinson_da

## Create tiles map

In [ ]:
valid_tiles_gdf = config.valid_tiles_gdf.to_crs("ESRI:54030")
valid_tiles_gdf

In [ ]:
# filter rows for where pix_ct_20XX for at least one year is greater than 0
valid_tiles_gdf = valid_tiles_gdf[valid_tiles_gdf.filter(like='pix_ct_20').gt(0).any(axis=1)]
valid_tiles_gdf

In [ ]:
f,ax=plt.subplots(figsize=(10, 10), subplot_kw={'projection': ccrs.Robinson()},dpi=300)
#ax.stock_img()
#ax.coastlines(linewidth=0.2)
global_hillshade_robinson_da.plot.imshow(ax=ax, transform=ccrs.Robinson(), cmap='gray', add_colorbar=False,alpha=1)
ax.set_title("")
valid_tiles_gdf[valid_tiles_gdf['success']==True].plot(ax=ax,transform=ccrs.Robinson(), color='none', edgecolor='black', linewidth=0.3)
ax.set_global()
# set background transparency
f.savefig('figures/valid_tiles_map.png', bbox_inches='tight', dpi=300, transparent=True, pad_inches=0)

## Create coarse global composites for methods figure

In [ ]:
coarsen_factor = 20
display_coarsen = 10

store = config.azure_blob_fs.get_mapper(f"snowmelt/snowmelt_runoff_onset/coarsened/global_{config.version}_coarsened_{coarsen_factor}_ds.zarr")
global_coarsened_ds = xr.open_zarr(store, consolidated=True, decode_coords='all',chunks="auto")#{'latitude':3*2048, 'longitude':3*2048}
global_coarsened_ds = global_coarsened_ds.rio.write_crs('EPSG:4326')


global_coarsened_ds = (
    global_coarsened_ds
    .coarsen(latitude=display_coarsen, longitude=display_coarsen, boundary="trim")
    .mean()
    .rio.write_crs("EPSG:4326")
    .compute()
)
global_coarsened_ds

In [ ]:
HS_KW   = dict(cmap="gray", add_colorbar=False, zorder=0, rasterized=True)
MEDIAN_KW = dict(cmap="viridis", vmin=110, vmax=270, add_colorbar=False) 
MAD_KW    = dict(cmap="Reds",    vmin=0,   vmax=30, add_colorbar=False)
TRES_KW   = dict(cmap="YlGn_r",  vmin=2,   vmax=14, add_colorbar=False)
all_vars_KW = [MEDIAN_KW, MAD_KW, TRES_KW]

In [ ]:
composite_vars = ["runoff_onset_median", "runoff_onset_mad", "temporal_resolution_median"]

for var, kw in zip(composite_vars, all_vars_KW):
    fig, ax = plt.subplots(figsize=(6, 3), subplot_kw={'projection': ccrs.Robinson()}, dpi=300)
    global_hillshade_robinson_da.plot.imshow(ax=ax, transform=ccrs.Robinson(), **HS_KW)
    global_coarsened_ds[var].plot.imshow(ax=ax, transform=ccrs.PlateCarree(), **kw)
    ax.set_title('')
    fig.savefig(f'figures/global_{var}_composite.png', bbox_inches='tight', dpi=300, transparent=True, pad_inches=0)

## Create coarse global maps for a single WY for methods figure

In [ ]:
WY_vars = ["runoff_onset", "temporal_resolution"]
WY_SELECTION_GLOBAL = 2020

for var, kw in zip(WY_vars, all_vars_KW):
    fig, ax = plt.subplots(figsize=(6, 3), subplot_kw={'projection': ccrs.Robinson()}, dpi=300)
    global_hillshade_robinson_da.plot.imshow(ax=ax, transform=ccrs.Robinson(), **HS_KW)
    global_coarsened_ds[var].sel(water_year=WY_SELECTION_GLOBAL).plot.imshow(ax=ax, transform=ccrs.PlateCarree(), **kw)
    ax.set_title('')
    fig.savefig(f'figures/global_{var}_{WY_SELECTION_GLOBAL}.png', bbox_inches='tight', dpi=300, transparent=True, pad_inches=0)

## Create coarse global maps of snow phenology for a single WY

In [ ]:
snow_phenology_vars = ["max_consec_snow_days", "SAD_DOWY", "SDD_DOWY"]
snow_phenology_vars_KW = [dict(cmap="Blues", add_colorbar=False),
                          dict(cmap="Purples_r", add_colorbar=False),
                          dict(cmap="Reds", add_colorbar=False)]

WY_SELECTION_GLOBAL = 2020

In [ ]:
snow_phenology_ds = xr.open_zarr(config.snow_phenology_store,
                                 decode_coords='all',
                                 consolidated=True,
                                 chunks='auto',
                                 )

snow_phenology_coarsened_ds = snow_phenology_ds.sel(water_year=WY_SELECTION_GLOBAL).coarsen(x=30, y=30, boundary="trim").mean().compute()
snow_phenology_coarsened_ds

In [ ]:
for var, kw in zip(snow_phenology_vars, snow_phenology_vars_KW):
    fig, ax = plt.subplots(figsize=(6, 3), subplot_kw={'projection': ccrs.Robinson()}, dpi=300)
    global_hillshade_robinson_da.plot.imshow(ax=ax, transform=ccrs.Robinson(), **HS_KW)
    snow_phenology_coarsened_ds[var].plot.imshow(ax=ax, transform=ccrs.Sinusoidal(), **kw)
    ax.set_title('')
    fig.savefig(f'figures/global_snow_phenology_{var}_{WY_SELECTION_GLOBAL}.png', bbox_inches='tight', dpi=300, transparent=True, pad_inches=0)

In [ ]:
# mock up a MOD10A2 scene by plotting the max_consec_snow_days just as before but with vmax=1
fig, ax = plt.subplots(figsize=(6, 3), subplot_kw={'projection': ccrs.Robinson()}, dpi=300)
global_hillshade_robinson_da.plot.imshow(ax=ax, transform=ccrs.Robinson(), **HS_KW)
snow_phenology_coarsened_ds["max_consec_snow_days"].where(lambda x: x >= 56).plot.imshow(ax=ax, transform=ccrs.Sinusoidal(), cmap="Blues", add_colorbar=False,vmin=56,vmax=56)
ax.set_title('')
fig.savefig(f'figures/global_snow_phenology_MOD10A2_{WY_SELECTION_GLOBAL}.png', bbox_inches='tight', dpi=300, transparent=True, pad_inches=0)

## Create Mt. Rainier runoff onset map for a single WY

In [ ]:
rainier_bbox_gdf = gpd.read_file("https://github.com/egagli/easysnowdata/raw/refs/heads/main/docs/examples/mt_rainier.geojson")
rainier_bbox_gdf

In [ ]:
rainier_x_slice = slice(585000,605000)
rainier_y_slice = slice(5.2E6,5.18E6)

In [ ]:
global_ds = xr.open_zarr(config.global_runoff_store, 
                        consolidated=True, 
                        decode_coords='all',
                        )


global_ds

In [ ]:
rainier_ds = global_ds.rio.clip_box(*rainier_bbox_gdf.total_bounds).compute().rio.reproject("EPSG:32610").sel(x=rainier_x_slice, y=rainier_y_slice).compute()
rainier_ds

In [ ]:
WY_SELECTION_RAINIER = 2023

In [ ]:
fig,ax=plt.subplots(figsize=(2,2),dpi=300)
rainier_ds['runoff_onset'].sel(water_year=WY_SELECTION_RAINIER).plot.imshow(ax=ax, **MEDIAN_KW)
ax.axis('off')
ax.set_title('')
fig.savefig(f'figures/rainier_runoff_onset_{WY_SELECTION_RAINIER}.png', bbox_inches='tight', pad_inches=0, dpi=300)    

In [ ]:
fig,ax=plt.subplots(figsize=(2,2),dpi=300)
rainier_ds['temporal_resolution'].sel(water_year=WY_SELECTION_RAINIER).plot.imshow(ax=ax, **TRES_KW)
ax.axis('off')
ax.set_title('')
fig.savefig(f'figures/rainier_temporal_resolution_{WY_SELECTION_RAINIER}.png', bbox_inches='tight', pad_inches=0, dpi=300)    

## Create Mt. Rainier snow phenology maps for a single WY

In [ ]:
snow_phenology_ds = xr.open_zarr(config.snow_phenology_store,
                                 decode_coords='all',
                                 consolidated=True,
                                 chunks='auto',
                                 ).astype(np.float16)
snow_phenology_ds

In [ ]:
rainier_snow_phenology_ds = snow_phenology_ds.rio.clip_box(*rainier_bbox_gdf.total_bounds, crs=rainier_bbox_gdf.crs).compute().astype("float32").rio.reproject("EPSG:32610").sel(x=rainier_x_slice, y=rainier_y_slice).compute()
rainier_snow_phenology_ds

In [ ]:
fig,ax=plt.subplots(figsize=(2,2),dpi=300)
rainier_snow_phenology_ds['max_consec_snow_days'].sel(water_year=WY_SELECTION_RAINIER).plot.imshow(cmap='Blues', ax=ax, add_colorbar=False)
ax.axis('off')
ax.set_title('')
fig.savefig(f'figures/rainier_max_consec_snow_days_{WY_SELECTION_RAINIER}.png', bbox_inches="tight",pad_inches=0, dpi=300)

fig,ax=plt.subplots(figsize=(2,2),dpi=300)
rainier_snow_phenology_ds['SAD_DOWY'].sel(water_year=WY_SELECTION_RAINIER).plot.imshow(cmap='Purples_r', ax=ax, add_colorbar=False)
ax.axis('off')
ax.set_title('')
fig.savefig(f'figures/rainier_SAD_dowy_{WY_SELECTION_RAINIER}.png', bbox_inches='tight', pad_inches=0, dpi=300)

fig,ax=plt.subplots(figsize=(2,2),dpi=300)
rainier_snow_phenology_ds['SDD_DOWY'].sel(water_year=WY_SELECTION_RAINIER).plot.imshow(cmap='Reds', ax=ax, add_colorbar=False)
ax.axis('off')
ax.set_title('')
fig.savefig(f'figures/rainier_SDD_dowy_{WY_SELECTION_RAINIER}.png', bbox_inches='tight', pad_inches=0, dpi=300)

## Create Sentinel-2 RGB map of Mt. Rainier

In [ ]:
s2 = easysnowdata.remote_sensing.Sentinel2(bbox_input=rainier_bbox_gdf,
                                           start_date="2020-08-14",
                                           end_date="2020-08-16",
                                           resolution=40,
                                           catalog_choice='planetarycomputer',            
                                           )
s2.get_rgb()
s2_rgb_da = s2.rgb_percentile.squeeze().compute().sel(x=rainier_x_slice, y=rainier_y_slice).compute()
s2_rgb_da

In [ ]:
fig,ax=plt.subplots(figsize=(2,2),dpi=300)
s2_rgb_da.plot.imshow(ax=ax,vmin=-0.05,vmax=0.5)
ax.axis('off')
ax.set_title('')
fig.savefig('figures/rainier_s2_rgb_2020-08-15.png', bbox_inches='tight', pad_inches=0, dpi=300)

## Create Sentinel-1 RTC images for different relative orbits

In [ ]:
s1_rtc_ds = easysnowdata.remote_sensing.Sentinel1(
    bbox_input=rainier_bbox_gdf,
    start_date="2020-01-01",
    end_date="2020-01-31",
    resolution=80,
)
s1_rtc_ds.data

s1_da = s1_rtc_ds.data['vv'].compute().sel(x=rainier_x_slice, y=rainier_y_slice).compute()
s1_da

In [ ]:
unique_orbits = np.unique(s1_da['sat:relative_orbit'])

for orbit_number in unique_orbits:
    fig,ax=plt.subplots(figsize=(2,2),dpi=300)
    s1_da[s1_da['sat:relative_orbit'] == orbit_number].isel(time=0).sel(x=rainier_x_slice,y=rainier_y_slice).plot.imshow(ax=ax, cmap='gray',vmin=-16,vmax=0,add_colorbar=False)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title('')#ax.set_title(f'Orbit {orbit_number}')
    fig.savefig(f'figures/s1_rtc_relorbit_{orbit_number}.png', dpi=300, bbox_inches='tight', pad_inches=0)


## Plot GMBA inventory

In [ ]:
# url = (f"https://data.earthenv.org/mountains/standard/GMBA_Inventory_v2.0_standard_300.zip")
# gmba_gdf = gpd.read_file("zip+" + url)
# gmba_gdf

# f,ax=plt.subplots(figsize=(10, 10), subplot_kw={'projection': ccrs.Robinson()},dpi=300)
# #ax.stock_img()
# #ax.coastlines(linewidth=0.2)
# global_hillshade_robinson_da.plot.imshow(ax=ax, transform=ccrs.Robinson(), cmap='gray', add_colorbar=False,alpha=1)
# gmba_gdf.plot(ax=ax, markersize=0.5, color='brown', alpha=0.7, transform=ccrs.PlateCarree())
# ax.set_title("")
# ax.set_global()
# # set background transparency
# f.savefig('figures/gmba_mountains.png', bbox_inches='tight', dpi=300, transparent=True)

# f,ax=plt.subplots(figsize=(10, 10), subplot_kw={'projection': ccrs.Robinson()},dpi=300)
# #ax.stock_img()
# #ax.coastlines(linewidth=0.2)
# global_hillshade_robinson_da.plot.imshow(ax=ax, transform=ccrs.Robinson(), cmap='gray', add_colorbar=False,alpha=1)
# gmba_gdf.plot(column='MapName',ax=ax, markersize=0.5, alpha=0.7, transform=ccrs.PlateCarree(), legend=False)
# ax.set_title("")
# ax.set_global()
# # set background transparency
# f.savefig('figures/gmba_mountains_diff_colors.png', bbox_inches='tight', dpi=300, transparent=True)